# 用 qust 写策略：表达式、回测和 K 线合成 block

这个 notebook 参考 `/root/qust-py/examples/demo.ipynb` 里的“写策略”模块，把策略写法拆成更清晰、可复制的几步。

qust 的核心思想是：策略不是一个和数据绑定在一起的大函数，而是一条 `Expr` 表达式链。表达式描述“怎么算”，数据源描述“从哪里来”。这样同一个策略可以放到：

- 历史 K 线回测；
- tick 数据合成 K 线后回测；
- 实盘增量数据流；
- 多品种 `.over("ticker")` 独立状态；
- 参数寻优或 monitor 可视化。

这和传统写法有明显区别。很多向量化策略会把全部历史数据一次性算完，写起来直观，但实盘里每来一条新数据就重算历史很浪费。事件驱动框架适合实盘，但策略代码往往变得很分散。qust 的目标是把表达式写法和流式状态保留结合起来：你仍然像写 DataFrame 一样组合算子，但底层算子可以维护状态。

本 notebook 主要展示四件事：

1. **K 线策略**：已经有 K 线时，直接写双均线信号；
2. **价格回测**：把目标持仓接到 `bt.price()`；
3. **K 线合成 block**：tick 数据先合成 1m/5m K 线；
4. **策略 block 嵌套**：在合成 K 线完成行上计算信号，再把信号回到 tick 流里。

本 notebook 使用本地样例数据，不依赖远程下载。


In [ ]:
import sys
sys.path.insert(0, "/root/otters/otters-py/python")

import importlib

import qust as qs
qs = importlib.reload(qs)


import qust.future.future  # 注册 kline / stra / bt / fp / monitor namespace
from qust import col
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(20)


## 1. 读取样例数据

这里用两类本地数据：

- `data_kline2.parquet`：已经是 K 线，可以直接写指标、信号和价格回测；
- `data_tick.parquet`：tick 数据，需要先合成 K 线，再把 K 线输出接到策略逻辑。

字段含义：

K 线数据：

- `ticker`: 品种；
- `datetime`: K 线时间；
- `open/high/low/close`: OHLC；
- `volume`: 成交量；
- `is_finished`: K 线是否完成。

tick 数据：

- `t`: tick 时间；
- `c`: 最新成交价或当前价格；
- `v`: 当前 tick 成交量；
- `bid1/ask1`: 一档买卖价；
- `bid1_v/ask1_v`: 一档买卖量；
- `ticker`: 品种。

后面的所有策略表达式都不直接写死数据变量。数据只在 `.calc_data(data)` 的时候传入。


In [27]:
KLINE_PATH = "/root/qust-py/examples/data/data_kline2.parquet"
TICK_PATH = "/root/qust-py/examples/data/data_tick.parquet"

raw_kline = pl.read_parquet(KLINE_PATH)
raw_tick = pl.read_parquet(TICK_PATH)

data_kline = raw_kline.filter(pl.col("ticker") == "au").head(5000)
data_tick = raw_tick.filter(pl.col("ticker") == "ag").head(12000)

print("raw kline shape:", raw_kline.shape)
print("sample kline shape:", data_kline.shape)
print("raw tick shape:", raw_tick.shape)
print("sample tick shape:", data_tick.shape)

data_kline.head(5)


raw kline shape: (610463, 8)
sample kline shape: (5000, 8)
raw tick shape: (1000000, 9)
sample tick shape: (12000, 9)


ticker,datetime,open,high,low,close,volume,is_finished
str,datetime[ms],f64,f64,f64,f64,f64,bool
"""au""",2022-07-02 00:01:00,390.160004,390.160004,390.059998,390.119995,81.0,true
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,390.079987,390.140015,52.0,true
"""au""",2022-07-02 00:03:00,390.140015,390.200012,390.119995,390.200012,50.0,true
"""au""",2022-07-02 00:04:01,390.200012,390.220001,390.140015,390.160004,61.0,true
"""au""",2022-07-02 00:05:00.500,390.140015,390.140015,390.079987,390.100006,41.0,true


## 2. K 线上的双均线策略

这一段展示最基础的策略表达式：双均线交叉。

表达式结构：

```python
col(
    "ticker",
    "datetime",
    "close",
    col("close").stra.two_ma(10, 20),
)
.with_cols(
    col("cross_up", "cross_down").stra.to_hold_always().expanding().alias("hold")
)
.over("ticker")
```

逐层解释：

1. `col("close").stra.two_ma(10, 20)`：计算短均线和长均线，并输出交叉信号；
2. `cross_up`: 短均线上穿长均线；
3. `cross_down`: 短均线下穿长均线；
4. `to_hold_always()`：把离散开仓信号转成持续持仓。上穿后持有多头，下穿后切换为空头或退出，具体语义由该策略算子定义；
5. `.expanding()`：这是有状态路径，表示持仓状态会随时间延续；
6. `.over("ticker")`：每个品种独立维护均线、交叉和持仓状态，避免不同品种的数据互相污染。

输出结果里同时保留了原始列、均线中间列、信号列和 `hold`，方便检查策略是不是按预期工作。


In [28]:
two_ma_strategy = (
    col(
        "ticker",
        "datetime",
        "close",
        col("close").stra.two_ma(10, 20),
    )
    .with_cols(
        col("cross_up", "cross_down")
        .stra
        .to_hold_always()
        .expanding()
        .alias("hold")
    )
    .over("ticker")
)

two_ma_signals = two_ma_strategy.calc_data(data_kline)
print("signals shape:", two_ma_signals.shape)
two_ma_signals.select(
    "ticker", "datetime", "close", "mean_short", "mean_long", "cross_up", "cross_down", "hold"
).head(20)


signals shape: (5000, 10)


ticker,datetime,close,mean_short,mean_long,cross_up,cross_down,hold
str,datetime[ms],f64,f64,f64,bool,bool,f64
"""au""",2022-07-02 00:01:00,390.119995,null,null,null,null,0.0
"""au""",2022-07-02 00:02:00.500,390.140015,null,null,null,null,0.0
"""au""",2022-07-02 00:03:00,390.200012,null,null,null,null,0.0
"""au""",2022-07-02 00:04:01,390.160004,null,null,null,null,0.0
"""au""",2022-07-02 00:05:00.500,390.100006,null,null,null,null,0.0
"""au""",2022-07-02 00:06:00.500,390.140015,null,null,null,null,0.0
"""au""",2022-07-02 00:07:00,389.880005,null,null,null,null,0.0
"""au""",2022-07-02 00:08:03,389.880005,null,null,null,null,0.0
…,…,…,…,…,…,…,…


## 3. 价格回测

有了目标持仓 `hold` 后，可以把它和价格一起送入回测算子：

```python
col("close", "hold").bt.price(fee_rate=0.0).expanding()
```

这一层的含义：

- `close`: 用作成交和盯市价格；
- `hold`: 当前目标持仓；
- `bt.price(...)`: 根据价格变化和持仓变化计算 PnL；
- `fee_rate`: 手续费率，这里设为 0 是为了让示例更容易看懂；
- `.expanding()`: 回测状态随着时间推进，不是每行孤立计算。

随后按日期汇总：

```python
col("pnl").sum().group_by(col("datetime").dt.date().alias("date"))
```

最后再做累计 PnL 并画线图。这个过程展示的是完整数据流：

```text
K 线输入 -> 策略信号 -> 持仓 -> PnL -> 日期汇总 -> 累计曲线 -> monitor line
```


In [29]:
pnl_expr = (
    two_ma_strategy
    .with_cols(
        col("close", "hold")
        .bt
        .price(fee_rate=0.0)
        .expanding()
    )
    .over("ticker")
    .select(
        col("pnl")
        .sum()
        .group_by(col("datetime").dt.date().alias("date"))
        .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
        .select("date", "pnl_cum")
    )
)

pnl_curve = pnl_expr.calc_data(data_kline)
print("pnl curve shape:", pnl_curve.shape)
pnl_curve.head(10)


pnl curve shape: (12, 2)


date,pnl_cum
date,f64
2022-07-02,0.299988
2022-07-04,-1.360077
2022-07-05,2.459839
2022-07-06,3.240021
2022-07-07,1.840027
2022-07-08,-0.340088
2022-07-09,-0.920074
2022-07-11,-3.779999
2022-07-12,-1.600006


In [30]:
pnl_plot = col("date", "pnl_cum").monitor("two_ma_pnl_curve", show_axis_label=True).line().runtime()
pnl_plot.plot(pnl_curve, open_in_jupyter=True, auto_open=False, height=520)


## 4. K 线合成 block：从 tick 合成 1m / 5m K 线

实际实盘里经常拿到的是 tick，而不是已经切好的 K 线。K 线合成应该作为策略表达式里的一块 block，而不是在策略外面临时预处理。这样回测和实盘可以复用同一条数据流。

输入 tick 列顺序固定为：

```python
col("t", "c", "v", "bid1", "ask1", "bid1_v", "ask1_v")
```

这一组列的含义是：

- `t`: tick 时间；
- `c`: 当前价格；
- `v`: 当前成交量；
- `bid1/ask1`: 一档盘口价格；
- `bid1_v/ask1_v`: 一档盘口量。

合成调用：

```python
col_tick.kline.rl1m.expanding()
col_tick.kline.rl5m.expanding()
```

重点是 `.expanding()`。K 线合成是有状态算子，它需要记住当前正在形成的 K 线：open、high、low、close、volume 和是否完成。没有 `.expanding()`，这个状态就没有正确的流式生命周期。

输出列包括：

- `datetime`: 合成 K 线时间；
- `open/high/low/close`: 合成 K 线 OHLC；
- `volume`: 合成 K 线成交量；
- `is_finished`: 当前输入 tick 是否让上一根 K 线完成。

这里同时合成 1 分钟和 5 分钟 K 线，并分别 `add_suffix("_1m")`、`add_suffix("_5m")`。当前 qust suffix 规则会产生 `close__1m`、`is_finished__1m` 这种双下划线列名，后面引用列时要按真实输出列名写。


In [31]:
col_tick = col("t", "c", "v", "bid1", "ask1", "bid1_v", "ask1_v")

kline_block_expr = (
    col(
        "ticker",
        col_tick.kline.rl1m.expanding().add_suffix("_1m"),
        col_tick.kline.rl5m.expanding().add_suffix("_5m"),
    )
    .over("ticker")
)

kline_blocks = kline_block_expr.calc_data(data_tick)
print("kline block shape:", kline_blocks.shape)
kline_blocks.select(
    "ticker",
    "datetime__1m", "open__1m", "high__1m", "low__1m", "close__1m", "volume__1m", "is_finished__1m",
    "datetime__5m", "close__5m", "is_finished__5m",
).tail(12)


kline block shape: (12000, 15)


ticker,datetime__1m,open__1m,high__1m,low__1m,close__1m,volume__1m,is_finished__1m,datetime__5m,close__5m,is_finished__5m
str,datetime[ms],f64,f64,f64,f64,f64,bool,datetime[ms],f64,bool
"""ag""",2024-01-06 02:23:46.500,5906.0,5906.0,5901.0,5903.0,571.0,false,2024-01-06 02:23:46.500,5903.0,false
"""ag""",2024-01-06 02:23:47,5906.0,5906.0,5901.0,5902.0,581.0,false,2024-01-06 02:23:47,5902.0,false
"""ag""",2024-01-06 02:23:47.500,5906.0,5906.0,5901.0,5902.0,583.0,false,2024-01-06 02:23:47.500,5902.0,false
"""ag""",2024-01-06 02:23:48,5906.0,5906.0,5901.0,5903.0,603.0,false,2024-01-06 02:23:48,5903.0,false
"""ag""",2024-01-06 02:23:48.500,5906.0,5906.0,5901.0,5903.0,610.0,false,2024-01-06 02:23:48.500,5903.0,false
"""ag""",2024-01-06 02:23:49,5906.0,5906.0,5901.0,5903.0,623.0,false,2024-01-06 02:23:49,5903.0,false
"""ag""",2024-01-06 02:23:49.500,5906.0,5906.0,5901.0,5903.0,624.0,false,2024-01-06 02:23:49.500,5903.0,false
"""ag""",2024-01-06 02:23:50.500,5906.0,5906.0,5901.0,5903.0,631.0,false,2024-01-06 02:23:50.500,5903.0,false
"""ag""",2024-01-06 02:23:51,5906.0,5906.0,5901.0,5904.0,632.0,false,2024-01-06 02:23:51,5904.0,false


In [32]:
completed_1m = (
    kline_blocks
    .filter(pl.col("is_finished__1m"))
    .select(
        pl.col("datetime__1m").alias("datetime"),
        pl.col("open__1m").alias("open"),
        pl.col("high__1m").alias("high"),
        pl.col("low__1m").alias("low"),
        pl.col("close__1m").alias("close"),
    )
    .tail(180)
)

print("completed 1m shape:", completed_1m.shape)
completed_1m.head(5)


completed 1m shape: (127, 5)


datetime,open,high,low,close
datetime[ms],f64,f64,f64,f64
2024-01-06 00:17:00,5933.0,5934.0,5932.0,5933.0
2024-01-06 00:18:00,5932.0,5932.0,5928.0,5929.0
2024-01-06 00:19:00,5929.0,5930.0,5928.0,5929.0
2024-01-06 00:20:00,5929.0,5930.0,5926.0,5927.0
2024-01-06 00:21:00,5926.0,5927.0,5924.0,5925.0


In [33]:
kline_plot = col("datetime", "open", "high", "low", "close").monitor(
    "synthetic_1m_kline",
    show_axis_label=True,
).kline().runtime()
kline_plot.plot(completed_1m, open_in_jupyter=True, auto_open=False, height=620)


## 5. 把 K 线合成 block 直接嵌进策略

这一节是整个 notebook 最重要的部分：策略不是先在 Python 外部把 tick 预处理成 K 线，再把 K 线 DataFrame 重新交给策略；而是把 K 线合成 block 直接嵌在 qust 表达式里。

数据流如下：

```text
tick 输入
-> col_tick.kline.rl1m.expanding()
-> 在完成的 1m K 线上计算 two_ma
-> 输出 cross_up__1m / cross_down__1m
-> to_hold_always().expanding()
-> 得到 tick 流上的 hold
```

为什么要 `filter_cb("is_finished")`：

- K 线合成过程中，大部分 tick 对应的 K 线还没完成；
- 如果每个未完成 tick 都更新均线，信号会在同一根 K 线内反复变化；
- `filter_cb("is_finished")` 表示只在 K 线完成时更新策略信号；
- 未完成 tick 上，持仓通过后面的状态算子延续。

为什么外层仍然 `.over("ticker")`：

- 不同品种有不同 tick 流；
- 每个品种的当前 K 线、均线和持仓都必须独立；
- `.over("ticker")` 把这条表达式链按品种拆开执行，然后再合并输出。

这个模式就是 demo 里“数据源不断获取多个品种 tick，策略需要分品种合成 K 线再生成信号”的核心写法。


In [34]:
tick_to_strategy = (
    col(
        "ticker",
        "t",
        "c",
        col_tick.kline.rl1m.with_cols(
            col("close").stra.two_ma(5, 10).filter_cb("is_finished")
        ).expanding().add_suffix("_1m"),
    )
    .with_cols(
        col("cross_up__1m", "cross_down__1m")
        .stra
        .to_hold_always()
        .expanding()
        .alias("hold")
    )
    .over("ticker")
)

tick_strategy_res = tick_to_strategy.calc_data(data_tick)
print("tick strategy shape:", tick_strategy_res.shape)
tick_strategy_res.select(
    "ticker",
    "t",
    "c",
    "close__1m",
    "is_finished__1m",
    "cross_up__1m",
    "cross_down__1m",
    "hold",
).tail(20)


tick strategy shape: (12000, 17)


ticker,t,c,close__1m,is_finished__1m,cross_up__1m,cross_down__1m,hold
str,datetime[ms],f64,f64,bool,bool,bool,f64
"""ag""",2024-01-06 02:23:42.500,5901.0,5901.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:43,5903.0,5903.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:43.500,5902.0,5902.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:44,5902.0,5902.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:44.500,5902.0,5902.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:45,5902.0,5902.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:45.500,5902.0,5902.0,false,null,null,-1.0
"""ag""",2024-01-06 02:23:46,5903.0,5903.0,false,null,null,-1.0
…,…,…,…,…,…,…,…


## 6. 实盘/回测复用同一策略表达式

回测时可以把一段历史数据一次性喂给 `calc_data`；实盘时可以把新来的 tick batch 一批批喂给同一个表达式对象。只要表达式对象不被丢弃，内部 `expanding()`、`rolling()`、K 线合成和持仓状态都会保留。

这和普通纯向量化 DataFrame 计算不同。普通写法通常是：

```text
每次来新数据 -> 拼接历史 -> 重新计算全部指标和策略
```

qust 的流式写法更接近：

```text
历史预热 -> 保留算子状态 -> 新 batch 到来 -> 只推进新增部分
```

这对于实盘很重要：

- 均线不用反复重算全部历史；
- K 线合成知道上一根未完成 K 线是什么；
- 持仓状态能从上一批延续到下一批；
- 回测和实盘可以用同一套表达式逻辑，减少两套代码不一致的问题。

下面用两段 tick 数据模拟“先历史预热，再来一批新数据”。注意：这里故意复用同一个 `live_expr` 对象连续调用两次 `calc_data`，就是为了展示状态保留。


In [35]:
live_expr = tick_to_strategy.select("ticker", "t", "hold")
history_batch = data_tick.head(8000)
next_batch = data_tick.slice(8000, 1000)

history_state = live_expr.calc_data(history_batch)
next_state = live_expr.calc_data(next_batch)

print("history output shape:", history_state.shape)
print("next output shape:", next_state.shape)
next_state.tail(10)


history output shape: (8000, 3)
next output shape: (1000, 3)


ticker,t,hold
str,datetime[ms],f64
"""ag""",2024-01-06 01:44:44.500,0.0
"""ag""",2024-01-06 01:44:45,0.0
"""ag""",2024-01-06 01:44:45.500,0.0
"""ag""",2024-01-06 01:44:46,0.0
"""ag""",2024-01-06 01:44:46.500,0.0
"""ag""",2024-01-06 01:44:47,0.0
"""ag""",2024-01-06 01:44:47.500,0.0
"""ag""",2024-01-06 01:44:48,0.0
"""ag""",2024-01-06 01:44:48.500,0.0
